# Maritime Perception MVP — Sea / Land / Vessel Segmentation

Cilj: brz, validacioni demo za semantičku segmentaciju maritimnih scena (more / kopno / nebo / prepreke) na bazi **MaSTr1325** dataset-a, koristeći pretrenirani backbone (transfer learning) za brz trening na Colab T4 GPU-u.

**Tok rada u ovom notebook-u:**
1. Setup okruženja
2. Preuzimanje MaSTr1325 dataset-a
3. Dataset / DataLoader
4. Model (DeepLabV3+ sa pretreniranim ResNet34 backbone-om)
5. Trening loop
6. Evaluacija (IoU po klasi)
7. Inference + vizuelizacija (overlay na slici/videu)
8. Export modela za demo aplikaciju (Gradio)

> Napomena: MaSTr1325 se preuzima sa zvaničnog izvora (https://box.vicos.si/borja/viamaro/index.html ili GitHub repo `bborja/mastr1325_dataset`). Ako link ne radi, alternativa je manji subset koji možeš ručno upload-ovati u Colab.

## 1. Setup

In [ ]:
!pip install -q segmentation-models-pytorch albumentations gradio opencv-python-headless

import os
import torch
import torch.nn as nn
import numpy as np
import cv2
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Idi na Runtime > Change runtime type > GPU pa pokreni ponovo.'

## 2. Preuzimanje MaSTr1325 dataset-a

MaSTr1325 ima 1325 anotiranih slika maritimnih scena sa 4 klase: **kopno (0), more (1), nebo (2), nepoznato/ignore (4)**, plus vessel/prepreke u nekim verzijama. Koristimo zvaničan GitHub repo sa download skriptom.

In [ ]:
!mkdir -p /content/data/mastr1325/all_images /content/data/mastr1325/all_masks

# Zvanični direktni linkovi (sa https://box.vicos.si/borja/viamaro/index.html):
!wget -q --show-progress -O /content/data/mastr1325_images.zip https://box.vicos.si/borja/mastr1325_dataset/MaSTr1325_images_512x384.zip
!wget -q --show-progress -O /content/data/mastr1325_masks.zip https://box.vicos.si/borja/mastr1325_dataset/MaSTr1325_masks_512x384.zip

!unzip -q /content/data/mastr1325_images.zip -d /content/data/mastr1325/all_images
!unzip -q /content/data/mastr1325_masks.zip -d /content/data/mastr1325/all_masks

print('Broj slika:', len(os.listdir('/content/data/mastr1325/all_images')))
print('Broj maski:', len(os.listdir('/content/data/mastr1325/all_masks')))

### 2b. Train/val split
Dataset dolazi kao jedan flat folder — pravimo 90/10 train/val split.

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

# Dataset dolazi kao flat folder (sve slike zajedno) - pravimo train/val split (90/10)
all_img_dir = '/content/data/mastr1325/all_images'
all_mask_dir = '/content/data/mastr1325/all_masks'

filenames = sorted([f for f in os.listdir(all_img_dir) if f.lower().endswith(('.jpg', '.png'))])
train_files, val_files = train_test_split(filenames, test_size=0.1, random_state=42)

for split_name, files in [('train', train_files), ('val', val_files)]:
    img_out = f'/content/data/mastr1325/{split_name}/images'
    mask_out = f'/content/data/mastr1325/{split_name}/masks'
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(mask_out, exist_ok=True)
    for fname in files:
        shutil.copy(os.path.join(all_img_dir, fname), os.path.join(img_out, fname))
        mask_name = os.path.splitext(fname)[0] + 'm.png'  # maske imaju 'm' dodato u nazivu
        mask_src = os.path.join(all_mask_dir, mask_name)
        if os.path.exists(mask_src):
            shutil.copy(mask_src, os.path.join(mask_out, mask_name))

print(f'Train: {len(train_files)} slika | Val: {len(val_files)} slika')

**Napomena o veličini:** Images zip je veći fajl (stotine MB), masks je mali. Ako Colab veza pretrgne preuzimanje, samo pokreni ćeliju ponovo (wget nastavlja gdje je stao samo ako dodaš `-c`, ili jednostavno ponovo pokreni od početka — fajlovi nisu preveliki za Colab disk).

**VAŽNO — stvarne klase u MaSTr1325 (3 klase + ignore region, NE 4 klase):**
- `0` = obstacles and environment (kopno, plovila, sve prepreke — **sve u jednoj klasi**)
- `1` = water (more)
- `2` = sky (nebo)
- `4` = ignore/unknown region (rubovi između klasa — isključuje se iz treninga, **ne** broji se kao posebna klasa za predikciju)

Plovila dakle **nisu posebna klasa** u ovom dataset-u — spadaju pod "obstacles". Ako želiš da posebno detektuješ plovila (vessel vs. land vs. drugi obstacle), to je sljedeća faza: trening dodatnog detekcionog modela (npr. YOLO) koji radi NA TOP segmentacione maske, fokusiran samo na regione klasifikovane kao "obstacle".

## 3. Dataset klasa i augmentacije

In [ ]:
# Stvarne MaSTr1325 klase u originalnim maskama: 0=obstacles/environment, 1=water, 2=sky, 4=ignore/unknown
# Plovila NISU posebna klasa - spadaju pod 'obstacles' zajedno sa kopnom i svim preprekama.
# Ignore region (4) se remapira na 255 i izuzima iz loss funkcije (ignore_index), kako autori preporučuju.
NUM_CLASSES = 3  # 0=obstacle/environment, 1=water, 2=sky
CLASS_NAMES = ['obstacle_environment', 'water', 'sky']
IGNORE_INDEX = 255
CLASS_COLORS = {
    0: (0, 0, 255),     # obstacle/environment (kopno+plovila+sve prepreke) - crveno (BGR za cv2)
    1: (255, 120, 0),   # water (more) - plavo/oranž
    2: (200, 200, 200), # sky (nebo) - sivo
}

def remap_mask(mask):
    """Original MaSTr1325 label 4 (ignore) -> 255 (ignore_index); ostalo ostaje isto."""
    mask = mask.copy()
    mask[mask == 4] = IGNORE_INDEX
    return mask

IMG_SIZE = 384

train_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class MaritimeSegDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.filenames = sorted(os.listdir(images_dir))
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img_path = os.path.join(self.images_dir, fname)
        mask_path = os.path.join(self.masks_dir, os.path.splitext(fname)[0] + 'm.png')  # 'm' u nazivu maske

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = remap_mask(mask)  # 4 -> 255 (ignore_index)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.long()

# Putanje prilagoditi nakon download-a (Korak 2)
TRAIN_IMAGES = '/content/data/mastr1325/train/images'
TRAIN_MASKS = '/content/data/mastr1325/train/masks'
VAL_IMAGES = '/content/data/mastr1325/val/images'
VAL_MASKS = '/content/data/mastr1325/val/masks'

# train_ds = MaritimeSegDataset(TRAIN_IMAGES, TRAIN_MASKS, transform=train_tfms)
# val_ds = MaritimeSegDataset(VAL_IMAGES, VAL_MASKS, transform=val_tfms)
# train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
# val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
print('Dataset klasa spremna. Otkomentariši gornje linije kad su putanje tačne.')

## 4. Model — DeepLabV3+ sa pretreniranim ResNet34

In [ ]:
model = smp.DeepLabV3Plus(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=NUM_CLASSES,
).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)  # rubni pikseli (4->255) se ne računaju u gubitku
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

print('Broj parametara modela:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')

## 5. Trening loop

In [ ]:
def iou_per_class(preds, targets, num_classes, ignore_index=255):
    ious = []
    preds = preds.view(-1)
    targets = targets.view(-1)
    valid = targets != ignore_index
    preds = preds[valid]
    targets = targets[valid]
    for c in range(num_classes):
        pred_c = (preds == c)
        target_c = (targets == c)
        intersection = (pred_c & target_c).sum().item()
        union = (pred_c | target_c).sum().item()
        ious.append(intersection / union if union > 0 else float('nan'))
    return ious

def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, loss_fn, num_classes):
    model.eval()
    total_loss = 0
    all_ious = []
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            all_ious.append(iou_per_class(preds, masks, num_classes))
    mean_ious = np.nanmean(np.array(all_ious), axis=0)
    return total_loss / len(loader), mean_ious

# EPOCHS = 30
# best_miou = 0
# for epoch in range(EPOCHS):
#     train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
#     val_loss, ious = validate(model, val_loader, loss_fn, NUM_CLASSES)
#     scheduler.step()
#     miou = np.nanmean(ious)
#     print(f'Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} mIoU={miou:.4f}')
#     print('  IoU po klasi:', dict(zip(CLASS_NAMES, [round(x,3) for x in ious])))
#     if miou > best_miou:
#         best_miou = miou
#         torch.save(model.state_dict(), '/content/best_model.pth')
#         print('  -> sačuvan novi najbolji model')
print('Trening loop definisan. Otkomentariši petlju kad su DataLoader-i spremni.')

## 6. Inference + vizuelizacija (overlay)

In [ ]:
def predict_and_overlay(model, image_bgr, alpha=0.5):
    """Uzima BGR sliku (cv2 format), vraća overlay sa segmentacionim maskama u boji."""
    model.eval()
    orig_h, orig_w = image_bgr.shape[:2]
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    augmented = val_tfms(image=image_rgb, mask=np.zeros(image_rgb.shape[:2], dtype=np.uint8))
    input_tensor = augmented['image'].unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        pred_mask = torch.argmax(output, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

    pred_mask_resized = cv2.resize(pred_mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

    color_mask = np.zeros_like(image_bgr)
    for class_id, color in CLASS_COLORS.items():
        color_mask[pred_mask_resized == class_id] = color

    overlay = cv2.addWeighted(image_bgr, 1 - alpha, color_mask, alpha, 0)
    return overlay, pred_mask_resized

# Test na jednoj slici:
# test_img = cv2.imread('/content/data/mastr1325/val/images/SAMPLE.jpg')
# overlay, mask = predict_and_overlay(model, test_img)
# plt.figure(figsize=(12,6))
# plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)); plt.title('Original'); plt.axis('off')
# plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); plt.title('Segmentacija'); plt.axis('off')
# plt.show()
print('Inference funkcija spremna.')

## 7. Video inference (opciono — za demo snimak)

In [ ]:
def process_video(model, input_path, output_path, alpha=0.5, max_frames=None):
    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        overlay, _ = predict_and_overlay(model, frame, alpha=alpha)
        out.write(overlay)
        frame_count += 1
        if max_frames and frame_count >= max_frames:
            break

    cap.release()
    out.release()
    print(f'Sačuvano: {output_path} ({frame_count} frejmova)')

# process_video(model, '/content/test_video.mp4', '/content/output_demo.mp4', max_frames=300)
print('Video inference funkcija spremna — koristi za snimak demo videa za pitch deck.')

## 8. Gradio demo aplikacija
Pokreni ovu ćeliju na kraju — daje ti link na web interfejs gdje upload-uješ sliku i vidiš rezultat uživo. Ovo je 'initial visualization of the product' koji ide u prijavu.

In [ ]:
import gradio as gr

def gradio_predict(input_image):
    image_bgr = cv2.cvtColor(np.array(input_image), cv2.COLOR_RGB2BGR)
    overlay, _ = predict_and_overlay(model, image_bgr)
    overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
    return overlay_rgb

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type='numpy', label='Maritimna scena (upload)'),
    outputs=gr.Image(type='numpy', label='Segmentacija: kopno/more/nebo/plovilo'),
    title='Maritime Perception MVP — Sea/Land/Vessel Segmentation',
    description='Demo percepcionog modela za maritimnu autonomiju. Upload-uj sliku scene sa mora.',
)

# demo.launch(share=True)  # share=True daje javni link koji možeš poslati drugima
print('Gradio app definisana. Pokreni demo.launch(share=True) kad je model istreniran.')

## Sljedeći korak nakon ovog notebook-a

1. Učitaj stvarni MaSTr1325 dataset (Korak 2) i otkomentariši DataLoader-e (Korak 3)
2. Pokreni trening (Korak 5) — na T4 GPU-u, ~30 epoha na 1325 slika traje tipično 20-40 min
3. Snimi kratak demo video sa overlay-em (Korak 7) — koristi za pitch deck
4. Pokreni Gradio app sa `share=True` i pošalji link kao 'live prototype' u prijavi
5. (Sljedeća faza, nakon ovog MVP-a) Dodaj detekciju plovila (YOLO) preko segmentacione maske kao drugi model — za 'characteristic model' sloj iz pipeline-a

## Zaključci i nalazi — testiranje generalizacije modela (Post-MVP analiza)

Nakon treniranja (mIoU = 0.989 na MaSTr1325 val setu) i testiranja na slikama van trening dataseta, primijećen je klasičan **domain shift** problem — model radi izvrsno na podacima slične distribucije kao trening set, ali ne generalizuje dobro na vizuelno različite scenarije. Detaljni nalazi po test slici:

### Test 1 — Slika u stilu MaSTr1325 (eye-level, USV kamera)
**Rezultat: skoro savršeno.** Segmentacija mora, neba i plovila (uključujući sitne detalje kao zastavice na jarbolima) je vrlo precizna i jasno prati granice objekata.

### Test 2 — Aerijalni/drone snimak marine (pogled odozgo, iz visine)
**Greške:**
- Veliko, otvoreno more je tačno klasifikovano (water/plavo)
- Nebo i kopno u pozadini (drveće, zgrade) pogrešno klasifikovano kao **obstacle** umjesto **sky**
- Sitne, svijetle/reflektujuće mrlje vode između trupova brodova pogrešno klasifikovane kao **sky** umjesto **water**

**Uzrok:** Trening dataset (MaSTr1325) sadrži isključivo snimke sa niske visine/eye-level perspektive, gdje nebo dominira gornjim dijelom slike kao veliki kontinuirani region. Iz drone ugla, horizont/nebo postaje mala, daleka traka koju model ne raspoznaje kao "svoju" klasu, dok sjajne sitne površine vode (nalik svjetlini neba u trening podacima) bivaju zamijenjene za nebo.

### Test 3 — Eye-level fotografija sa čistim, intenzivno plavim nebom (bez oblaka)
**Greška:** Cijelo čisto plavo nebo klasifikovano kao **water** umjesto **sky**, dok su brodovi i stvarna voda ispod njih tačno klasifikovani.

**Uzrok:** MaSTr1325 trening slike pretežno sadrže maglovito/oblačno, bjelkasto-sivo nebo (tipično za snimanje na nivou mora uz vlažnu atmosferu). Čisto, zasićeno plavo nebo bez oblaka je vizuelno blisko boji mirne vode u trening distribuciji, pa model nema dovoljno primjera da nauči razliku — kad vidi intenzivno plavu boju, "default" klasifikacija postaje water (dominantnija klasa sa tom bojom u trening podacima).

### Ključni obrazac kroz sva tri testa

| Klasa | Robusnost na nove scenarije |
|---|---|
| **Plovila / obstacle** | Konzistentno dobra detekcija u sva tri testa — najbitnija klasa za bezbjednost (kolizija, navigacija) pokazuje najveću robusnost |
| **More / water** | Dobra u kontekstu velikih, kontinuiranih regiona; greši na rubnim slučajevima (sitne reflektujuće površine, drugi ugao kamere) |
| **Nebo / sky** | Najmanje robusna klasa — osjetljiva na promjene atmosferskih uslova (jasno vs. maglovito nebo) i ugla kamere (eye-level vs. aerijalno) |

### Implikacije za stvarnu primjenu

Ovaj nalaz je direktno relevantan za probleme opisane u oglasima za posao u maritimnoj AI percepciji (npr. Tocaro Blue/ProteusCore) — gdje se naglašava da "off-the-shelf modeli ne uspijevaju" u dinamičnim maritimnim uslovima. Konkretno:

1. **Diverzitet trening podataka po uglovima kamere i atmosferskim uslovima** je kritičan za robusnu generalizaciju — ovo se direktno odnosi na "design-of-experiment methods for data collection and annotation" pomenute u opisima ovakvih pozicija
2. **Prioritizacija klasa po bezbjednosnom značaju** — sistem koji pouzdano detektuje plovila/prepreke (kritično za izbjegavanje kolizije) ali griješi na nebu (manje kritično) je prihvatljiviji kompromis nego obrnuto
3. **Sljedeći korak za poboljšanje**: proširiti trening dataset sa slikama iz više izvora (drone snimci, različiti atmosferski uslovi, čisto vs. oblačno nebo) ili primijeniti domain adaptation tehnike pre nego što se sistem osloni isključivo na MaSTr1325 distribuciju.
